# g1_limpo — treino DO ZERO na Kaggle (`zero06`)

Linhagem NOVA: pesos aleatórios, sem checkpoint de entrada, sem retomada. A sessão roda o
que couber em `HORAS_LIMITE` e para.

**Antes de rodar:**
1. **Settings → Accelerator → GPU** (T4 ×1 ou P100).
2. **Settings → Internet → On.** O clone do GitHub e o upload do checkpoint precisam dela.
3. **NÃO anexe dataset de entrada.** Do zero não há o que ler.
4. A branch tem de estar no GitHub: `git push -u origin exp/g1-limpo-v2`.

⚠ Este notebook não continua nada. Quem continua uma linhagem é o
`g1_limpo_kaggle.ipynb`, que lê o `model_*.pt` de maior número do dataset
`g1-limpo-zero`.

⚠ A `zero03` (iteração 451) e a `zero04` (iteração 650) morreram com o
`batente_da_cintura` dominante: a terminação limitava o custo do batente em −4 e
encerrava o episódio antes de a rampa (53,6/s) agir, e todo episódio de manipulação
acabava em ~7 passos. A terminação SAIU deste notebook; a rampa fica.

In [ ]:
# ⚠ NADA DE `import torch` aqui: ele registra operadores C++ no import, e se entrar antes
# do pip um reload depois levanta `Only a single TORCH_LIBRARY ... triton`.
import subprocess, sys
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU: Settings -> Accelerator -> GPU"

In [ ]:
# ⚠ LISTA DE ARGUMENTOS, nunca string de shell: `numpy<2.5` viraria redirecionamento e o
# pip não rodaria. Só `mjlab` — ele pina a árvore inteira (mujoco-warp, rsl-rl-lib 5.4.0,
# numpy<2.5). `torch` NÃO entra: trocá-lo perde a GPU sem avisar. 1.5.3, não 1.5.1: o
# `RslRlModelCfg` de 1.5.1 manda `cnn_cfg`/`rnn_type` que o `MLPModel` de 5.4.0 rejeita.
import subprocess, sys
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "mjlab==1.5.3"]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])
assert r.returncode == 0, "o pip falhou. Confira Settings -> Internet -> On"

In [ ]:
# ⚠ Se o pip trocou o torch por um build sem CUDA, tudo abaixo roda em CPU sem reclamar.
# A checagem vai num subprocesso porque `reload(torch)` não existe.
import subprocess, sys
chk = subprocess.run([sys.executable, "-c",
    "import torch;print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-400:])
assert " True" in chk.stdout, "o pip levou a CUDA embora"

import torch, mjlab, mujoco
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}  |  mujoco {mujoco.__version__}")

In [ ]:
import importlib, os, pathlib, shutil, subprocess, sys
os.environ.setdefault("MUJOCO_GL", "egl")

RUN    = "zero06"              # ⚠ NOME NOVO A CADA MUDANÇA DE CONJUNTO. Ele é a impressão
                               # digital da tabela de recompensa, e mantém duas tabelas
                               # diferentes fora do MESMO gráfico.
BRANCH = "exp/g1-limpo-v2"

# ⚠ TUDO EM `/kaggle/working`: é o único disco desta sessão, e morre com ela se você
# não comitar nem subir o checkpoint pela API (célula do fim).
BASE     = pathlib.Path("/kaggle/working")
RAIZ     = BASE / "g1"
LOG_ROOT = BASE / "logs"
raiz_exp = LOG_ROOT / "g1_limpo"

# ⚠ Se você deu `git push` depois de abrir esta sessão, RE-RODE esta célula: a Kaggle
# já rodou código antigo sem avisar uma vez.
if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)], check=True)
print("clone =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` não é higiene: o Python cacheia um finder POR DIRETÓRIO, e o de um
# diretório que não existia na inserção fica cacheado como VAZIO.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

LOG_ROOT.mkdir(parents=True, exist_ok=True)
if sorted(raiz_exp.glob(f"*_{RUN}")):
    print(f"\n⚠⚠ JÁ EXISTE run de {RUN!r}: rodar o treino de novo começa OUTRA VEZ do zero,"
          " numa pasta nova, e não continua esta.")
print("log_root =", LOG_ROOT, "| vazio:", not list(raiz_exp.rglob("*")))

In [ ]:
# =====================================================================
#  SAÍDA — sobe o checkpoint pela API da Kaggle. Chamada num `finally`, célula do treino.
# =====================================================================
# ⚠ Persistência sem você presente: uma sessão morta (12 h, ociosidade, aba fechada)
# leva `/kaggle/working` inteiro. `Save Version -> Save & Run All (Commit)` também
# persiste, mas exige commit; isto funciona numa sessão interativa comum.
#
# Precisa de três coisas, uma vez só:
#   1. Add-ons -> Secrets -> crie `KAGGLE_USERNAME` e `KAGGLE_KEY` (valores em
#      https://www.kaggle.com/settings -> API -> Create New Token).
#   2. Settings -> Internet -> On.
#   3. O dataset `g1-limpo-zero` já existe (crie-o uma vez, vazio, em
#      https://www.kaggle.com/datasets -> New Dataset, título `G1-Limpo-Zero`).
#
# ⚠⚠ O SLUG É `g1-limpo-zero`, NUNCA `g1-limpo-v2`. O `g1_limpo_kaggle.ipynb` acha o
# `model_*.pt` de MAIOR número em TODO o `/kaggle/input`, sem olhar de que dataset veio;
# subir aqui no dataset errado faria a próxima sessão de continuação achar um checkpoint
# de outra linhagem, sem aviso.
import json, os, pathlib, re, shutil, subprocess

SLUG = "g1-limpo-zero"
ULTIMO_CKPT = None


def _run_nova():
    # a pasta de run mais recente desta `RUN`, e seus `model_*.pt` ordenados por número
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir() if p.is_dir() and p.name.endswith(RUN))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks


def sobe_checkpoint():
    """Zipa nada: sobe o `.pt` CRU (a origem do `g1_limpo_kaggle.ipynb` acha por `rglob`)."""
    global ULTIMO_CKPT
    nova, cks = _run_nova()
    if not cks:
        print("nada para subir: o treino não salvou checkpoint")
        return
    ULTIMO_CKPT = cks[-1]

    envio = BASE / "envio"
    shutil.rmtree(envio, ignore_errors=True)
    envio.mkdir()
    shutil.copy2(ULTIMO_CKPT, envio / ULTIMO_CKPT.name)
    dig = raiz_exp / f"{RUN}.pesos.json"
    if dig.exists():
        shutil.copy2(dig, envio / dig.name)

    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
    usuario = os.environ["KAGGLE_USERNAME"]
    (envio / "dataset-metadata.json").write_text(json.dumps(
        {"title": "G1-Limpo-Zero", "id": f"{usuario}/{SLUG}",
         "licenses": [{"name": "CC0-1.0"}]}, indent=1))

    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(envio),
                        "-m", f"{RUN} it{int(re.search(r'(\d+)', ULTIMO_CKPT.name).group(1))}",
                        "-r", "skip"], capture_output=True, text=True)
    saida = (r.stdout or "") + (r.stderr or "")
    print(saida)
    if r.returncode != 0:
        print(f"⚠⚠ UPLOAD FALHOU. O checkpoint ainda está em disco: {ULTIMO_CKPT}")
        print(f"   Baixe pelo painel Data -> Output antes de fechar a aba.")
        if "404" in saida or "not found" in saida.lower():
            print(f"   CAUSA PROVÁVEL: o dataset {usuario}/{SLUG} não existe — crie-o "
                  "vazio em kaggle.com/datasets (título G1-Limpo-Zero) e rode de novo.")
        return
    print(f"subiu {ULTIMO_CKPT.name} em {usuario}/{SLUG}")

In [ ]:
# TENSORBOARD AO VIVO — rode ANTES do treino.
# ⚠ A célula do treino BLOQUEIA o notebook, mas este painel continua se atualizando
#   sozinho enquanto ela roda.
# ⚠ O log ainda não existe agora: o painel abre em "No dashboards are active" e se
#   preenche na primeira escrita do runner. Use o botão de recarregar DELE — re-rodar
#   esta célula abre uma segunda instância.
%load_ext tensorboard
%tensorboard --logdir $LOG_ROOT --reload_interval 30

## O treino

Três diferenças em relação ao notebook de continuação: a tabela do `limite_de_junta` é a
**do zero** (a rampa acaba NO batente, e não em 155% do curso), `max_iterations` vem só do
relógio, e a LR fica no default de 1e-3 — baixar para 5e-4 é regra de warm-start, e aqui
não há política velha para desmanchar.

In [ ]:
import dataclasses, inspect, json, sys
sys.path.insert(0, str(RAIZ))
import g1_limpo
from g1_limpo import comando as CMD, knobs as KN

from mjlab.scripts.train import TrainConfig, launch_training

NUM_ENVS     = 4096            # ⚠ pelo HOSPEDEIRO, não pela VRAM: medido, a GPU da Kaggle
                               # não roda 8192 com os mesmos 16 GB que o Colab roda.
HORAS_LIMITE = 10.5            # a sessão da Kaggle é dura em 12 h; a margem é o pip, o
                               # clone e a montagem do env.
SEG_POR_ITER = 6.9             # ⚠ MEDIDO a 4096 envs, 1× T4: 14 265 passos/s com 98 304
                               # passos por iteração. Corrija com o `Collection time` real
                               # do primeiro log.

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID), log_root=str(LOG_ROOT))
cfg.env.scene.num_envs   = NUM_ENVS
cfg.agent.run_name       = RUN
cfg.agent.logger         = "tensorboard"
cfg.agent.max_iterations = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)

# ------------------------------------------------- a tabela DO ZERO do limite_de_junta
# ⚠ A rampa acaba NO BATENTE. A tabela larga do knob acaba em 155% do meio-curso: ela
# conserta política viciada e ACEITA o batente como preço. Do zero o robô tem de aprender
# a não chegar nele.
# ⚠⚠ A CINTURA tem tripla própria: o `waist_pitch` tem só 60° de curso e a referência da
# IK o põe em frac 0,900 — com limiar 0,85 esta rampa cobrava A PRÓPRIA REFERÊNCIA, 1,72
# por passo. Com 0,90 a referência cai no início da rampa, de graça; o pico não muda —
# `expm1(40 × 0,10) = 53,6`, por segundo, no batente.
# ⚠⚠ SEM TERMINAÇÃO NO BATENTE. A `zero03` (it 451) e a `zero04` (it 650) tinham
# `NoBatente` sobre a cintura em `frac_max = 1,00`: a ação MÉDIA levava o `waist_pitch`
# ao batente em 7 passos com o comando parado (MEDIDO, ator determinístico), a
# terminação limitava o custo disso em −4 uma vez e encerrava o episódio ANTES da rampa
# agir — 53,6/s nunca foi cobrado. Todo episódio de manipulação morria na espera
# inicial. Sem a terminação o robô fica vivo, paga a rampa e recebe o gradiente dos
# outros termos.
_lj = dataclasses.replace(KN.LimiteDeJunta(),
                          tornozelo=(20.0, 0.15, 0.85), punho=(20.0, 0.15, 0.85),
                          resto=(20.0, 0.15, 0.85),     cintura=(40.0, 0.10, 0.90),
                          hip_yaw=(20.0, 0.15, 0.25))
cfg.env.rewards["limite_de_junta"].params["tabela"] = _lj.por_padrao()

# ⚠ O FREIO A −6 NO ZERO, e não o −15 do knob (22/09). MEDIDO na `zero05`: com o std
# perto de 1 o −15 custava −14,8/s, 64% dos custos ligados ao ruído; o std caiu a 0,45
# contra 0,61 da task de fábrica, e com o robô de pé a taxa líquida ficava em ~0/s. O −2
# da `zero01` deixava o robô se mexer rápido demais. O knob segue em −15.
cfg.env.rewards["velocidade_por_regime"].weight = -6.0

cfg.env.sim.mujoco.cone     = "pyramidal"   # ⚠ "elliptic" já divergiu para NaN duas vezes
cfg.env.sim.mujoco.impratio = 2.0

# ------------------------------------------------- o clone é o que eu penso que é?
# ⚠ Poucos asserts, e cada um pega um modo de falha SILENCIOSO: clone velho que treina
# dez horas com a tabela errada e não avisa.
rw, tm, cu = cfg.env.rewards, cfg.env.terminations, cfg.env.curriculum
_tab = rw["limite_de_junta"].params["tabela"]
assert len(_tab) == 14 and all(len(v) == 3 for v in _tab.values()), \
    f"clone anterior à tripla `(k, teto, limiar)`: {_tab}"
# ⚠ SEM CONTAGEM DE TERMOS: o número real sobe toda vez que um termo legítimo entra, e a
# contagem não discrimina clone velho, só cria manutenção. A ORDEM discrimina: a
# `renda_congelada` lê o `_step_reward` dos outros, e tem de ser avaliada por último.
assert list(rw)[-1] == "renda_congelada", \
    f"a última recompensa é {list(rw)[-1]!r}, e tem de ser a `renda_congelada`: ela lê o " \
    "`_step_reward` das outras e leria zero se corresse antes"
assert list(cu) == ["command_vel", "forma", "nivel", "elo"], \
    f"ordem do currículo errada ({list(cu)}): `forma` e `nivel` leriam o elo do episódio SEGUINTE"
assert sorted(tm) == ["caixa_largada", "fell_over", "time_out"], sorted(tm)
assert hasattr(CMD, "FACE_DE_PE"), \
    "clone anterior a 21/09: o BOTAR ainda congela a face na caixa TOMBADA nas mãos, e o " \
    "`alinhado` passa a exigir que ela MANTENHA o tombo"
assert "FACE_CONGELADA" in inspect.getsource(CMD.AlvoCaixaCmd._recalcula_sigmas), \
    "clone anterior a 22/09: o `sigma_ori` do PEGAR ainda vem de um erro que é ZERO por " \
    "construção, e tombar a caixa ao erguê-la sai de graça"
assert {"caixa_na_pega", "caixa_no_botar"} <= set(cfg.env.metrics), \
    "sem as duas sentinelas não dá para ver de qual elo vem o tombo da caixa"
assert cfg.agent.algorithm.learning_rate == 1.0e-3 and cfg.agent.seed == 42
assert not str(LOG_ROOT).startswith(str(RAIZ)), "log dentro do clone: o re-clone o apaga"

print(f"envs {NUM_ENVS} | lote {NUM_ENVS * cfg.agent.num_steps_per_env} | "
      f"{cfg.agent.max_iterations} iterações | LR {cfg.agent.algorithm.learning_rate}")
print("[CONFERE] ok. começando do zero.\n")

raiz_exp.mkdir(parents=True, exist_ok=True)
(raiz_exp / f"{RUN}.pesos.json").write_text(json.dumps(
    {k: float(v.weight) for k, v in rw.items()}, indent=1, sort_keys=True))

try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    sobe_checkpoint()

## O que olhar no log

| iteração | canal | o que responde |
|---|---|---|
| ~400 | `fell_over` e `postura_ereta` | ele fica de pé? |
| ~1000 | `Metrics/twist/razao_marcha` | passou de 0,50? é a marcha |
| ~2500 | `Curriculum/forma/sorteio` | desceu de 0,95? a manipulação entrou |
| sempre | `Episode_Metrics/caixa_na_pega` | graus de tombo ao SEGURAR. Tem de cair; o fecho exige 25° |
| sempre | `Episode_Metrics/caixa_no_botar` | graus de tombo ao POUSAR. Mesma régua |
| sempre | `Curriculum/forma/s_B` e `s_C` | as taxas de fecho do PEGAR e do BOTAR |
| sempre | `Episode_Metrics/velocidade_de_junta` | o freio de segurança; era 2,0 antes do peso −15 |
| a 1ª | `Collection time` | corrija o `SEG_POR_ITER` com o valor real |

**Continuar depois:** a célula do fim sobe o `model_*.pt` e o `.pesos.json` para o
dataset `g1-limpo-zero`. Se a sessão caiu antes disso, baixe pelo painel **Data →
Output**. Abra o `g1_limpo_kaggle.ipynb`, anexe **só** o dataset `g1-limpo-zero`
(nunca junto com `g1-limpo-v2` — a origem de lá acha o `model_*.pt` de maior número em
TODO `/kaggle/input`) e troque o `RUN` de lá para `zero06`.